In [15]:
%pip install python-dotenv transformers peft datasets langchain langchain-groq langchain-google-genai accelerate
# bitsandbytes is for quantization on GPU; on CPU we simulate the math instead
# %pip install bitsandbytes --quiet  # uncomment if on a GPU machine

In [16]:
from dotenv import load_dotenv
import os, pathlib, getpass

# Load from unit 2/.env (shared key store for this course)
# Works regardless of which directory the notebook is launched from
_env_path = pathlib.Path(__file__).parent.parent / 'unit 2' / '.env' if '__file__' in dir() else pathlib.Path('../unit 2/.env')
load_dotenv(_env_path)

# Fallback: prompt if key is still missing (e.g. first-time setup or revoked key)
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API Key not found — enter it now: ")

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key not found — enter it now: ")

print(f"GOOGLE key loaded: {'yes' if os.getenv('GOOGLE_API_KEY') else 'NO — check unit 2/.env'}")
print(f"GROQ   key loaded: {'yes' if os.getenv('GROQ_API_KEY') else 'NO — check unit 2/.env'}")

GOOGLE key loaded: yes
GROQ   key loaded: yes



## Part 3a: Fine-Tuning

In [17]:
import torch
import numpy as np

# ---- LoRA Math Demonstration ----
# We simulate a tiny LoRA on a 64x64 weight matrix

torch.manual_seed(42)

d = 64    # hidden dimension (in real LLMs this is 4096)
r = 4     # LoRA rank (in real use: 4, 8, 16, 32)

# Original frozen weight matrix (would be the Q/K/V projection in attention)
W = torch.randn(d, d) * 0.01
W.requires_grad_(False)  # Frozen!

# LoRA adapter matrices
A = torch.randn(r, d) * 0.01  # shape: (r, d)
B = torch.zeros(d, r)          # shape: (d, r) — initialized to zero so ΔW=0 at start
A.requires_grad_(True)
B.requires_grad_(True)

# Scaling factor (alpha/r — controls magnitude of LoRA contribution)
alpha = 16
scaling = alpha / r

# Forward pass
x = torch.randn(1, d)
h_original = x @ W.T                     # Standard: x * W
h_lora     = x @ W.T + (x @ A.T @ B.T) * scaling  # LoRA: x*W + x*A*B (scaled)

# Parameter counts
total_params     = d * d
lora_params      = r * d + d * r  # A + B
pct              = 100 * lora_params / total_params

print(f"Original weight W:     {d}x{d} = {total_params:,} parameters")
print(f"LoRA matrices A+B:     {r}x{d} + {d}x{r} = {lora_params:,} parameters")
print(f"LoRA overhead:         {pct:.2f}% of original")
print(f"")
print(f"Original output:  {h_original[0, :4].detach().numpy().round(5)}")
print(f"LoRA output:      {h_lora[0, :4].detach().numpy().round(5)}")
print(f"(outputs identical at init because B=0, so ΔW = B@A = 0)")

Original weight W:     64x64 = 4,096 parameters
LoRA matrices A+B:     4x64 + 64x4 = 512 parameters
LoRA overhead:         12.50% of original

Original output:  [ 0.0417  -0.05841  0.07145  0.00424]
LoRA output:      [ 0.0417  -0.05841  0.07145  0.00424]
(outputs identical at init because B=0, so ΔW = B@A = 0)


In [18]:
# ---- PEFT with HuggingFace (conceptual demo) ----
# This shows the actual API used in production fine-tuning jobs

from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading a tiny model for PEFT demo (distilgpt2)...")
base_model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# Define LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # Causal Language Modeling
    r=8,                             # Rank — larger r = more capacity, more params
    lora_alpha=16,                   # Scaling: effective_scale = lora_alpha / r = 2.0
    lora_dropout=0.1,                # Regularization
    target_modules=["c_attn"],       # Apply LoRA only to attention projections
    bias="none"                      # Don't train bias terms
)

# Wrap the base model with LoRA adapters
peft_model = get_peft_model(base_model, lora_config)

# Compare parameter counts
peft_model.print_trainable_parameters()

Loading a tiny model for PEFT demo (distilgpt2)...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [19]:
# What does the LoRA model look like?
for name, module in peft_model.named_modules():
    if "lora" in name.lower() and "weight" in name.lower():
        print(f"  {name}: {tuple(module.shape) if hasattr(module, 'shape') else type(module)}")

## Part 3b: Data Resolution & Precision

In [20]:
import torch
import struct

# ---- Numerical Precision Demonstration ----

value = 1.3456789

fp32 = torch.tensor(value, dtype=torch.float32)
fp16 = torch.tensor(value, dtype=torch.float16)
bf16 = torch.tensor(value, dtype=torch.bfloat16)

print(f"Original value:  {value}")
print(f"FP32:            {fp32.item():.10f}  (error: {abs(fp32.item() - value):.2e})")
print(f"FP16:            {fp16.item():.10f}  (error: {abs(fp16.item() - value):.2e})")
print(f"BF16:            {bf16.item():.10f}  (error: {abs(bf16.item() - value):.2e})")

print("\n--- Memory footprint of a 1-billion parameter model ---")
params = 1_000_000_000
for name, bits in [("FP32", 32), ("FP16/BF16", 16), ("INT8", 8), ("INT4", 4)]:
    gb = params * bits / (8 * 1024**3)
    print(f"  {name:<12}: {gb:.2f} GB")

Original value:  1.3456789
FP32:            1.3456789255  (error: 2.55e-08)
FP16:            1.3457031250  (error: 2.42e-05)
BF16:            1.3437500000  (error: 1.93e-03)

--- Memory footprint of a 1-billion parameter model ---
  FP32        : 3.73 GB
  FP16/BF16   : 1.86 GB
  INT8        : 0.93 GB
  INT4        : 0.47 GB


In [21]:
# ---- FP16 Overflow Danger Demonstration ----

print("FP16 overflow example:")
big_value = 70000.0  # Within FP32 range, outside FP16 range (max: 65504)
fp32_big  = torch.tensor(big_value, dtype=torch.float32)
fp16_big  = torch.tensor(big_value, dtype=torch.float16)
bf16_big  = torch.tensor(big_value, dtype=torch.bfloat16)

print(f"  Value:   {big_value}")
print(f"  FP32:    {fp32_big.item()}  (correct)")
print(f"  FP16:    {fp16_big.item()}  ← OVERFLOW! Becomes inf")
print(f"  BF16:    {bf16_big.item()}  (safe — wide exponent range)")

FP16 overflow example:
  Value:   70000.0
  FP32:    70000.0  (correct)
  FP16:    inf  ← OVERFLOW! Becomes inf
  BF16:    70144.0  (safe — wide exponent range)


## Part 3c: Quantization

In [22]:
import numpy as np

def quantize_to_int8(weights: np.ndarray):
    """
    Quantize FP32 weights to INT8 using linear (affine) quantization.
    Returns: (quantized_weights, scale, zero_point)
    """
    x_min, x_max = weights.min(), weights.max()
    q_min, q_max = -128, 127

    S = (x_max - x_min) / (q_max - q_min)           # scale
    Z = round(q_min - x_min / S)                      # zero-point
    Z = int(np.clip(Z, q_min, q_max))

    q = np.round(weights / S + Z).astype(np.int8)    # quantize
    return q, S, Z

def dequantize(q: np.ndarray, S: float, Z: int) -> np.ndarray:
    """Reconstruct FP32 values from INT8 quantized weights."""
    return S * (q.astype(np.float32) - Z)

# Simulate a small weight matrix (like one row of an attention projection)
np.random.seed(42)
fp32_weights = np.random.randn(8).astype(np.float32)  # FP32 weights

int8_weights, S, Z = quantize_to_int8(fp32_weights)
reconstructed     = dequantize(int8_weights, S, Z)
error             = np.abs(fp32_weights - reconstructed)

print("Linear Quantization Demo (FP32 → INT8 → FP32)")
print("=" * 65)
print(f"Scale factor S = {S:.6f}")
print(f"Zero point   Z = {Z}")
print()
print(f"{'Index':<8} {'FP32':<12} {'INT8':<10} {'Reconstructed':<16} {'Error'}")
print("-" * 65)
for i in range(len(fp32_weights)):
    print(f"  [{i}]   {fp32_weights[i]:>10.6f}   {int8_weights[i]:>5}   {reconstructed[i]:>14.6f}   {error[i]:.6f}")

print(f"\nMax quantization error: {error.max():.6f}")
print(f"Mean quantization error: {error.mean():.6f}")

# Memory comparison
fp32_bytes = fp32_weights.nbytes
int8_bytes = int8_weights.nbytes
print(f"\nMemory: FP32={fp32_bytes} bytes → INT8={int8_bytes} bytes ({fp32_bytes/int8_bytes:.1f}x reduction)")

Linear Quantization Demo (FP32 → INT8 → FP32)
Scale factor S = 0.007111
Zero point   Z = -95

Index    FP32         INT8       Reconstructed    Error
-----------------------------------------------------------------
  [0]     0.496714     -25         0.497787   0.001073
  [1]    -0.138264    -114        -0.135114   0.003151
  [2]     0.647689      -4         0.647123   0.000566
  [3]     1.523030     119         1.521805   0.001225
  [4]    -0.234153    -128        -0.234671   0.000518
  [5]    -0.234137    -128        -0.234671   0.000534
  [6]     1.579213     127         1.578695   0.000518
  [7]     0.767435      13         0.768014   0.000579

Max quantization error: 0.003151
Mean quantization error: 0.001020

Memory: FP32=32 bytes → INT8=8 bytes (4.0x reduction)


In [23]:
# ---- Model Size Comparison: FP32 vs INT8 vs INT4 ----

print("Model Size Comparison for Common LLMs")
print("=" * 65)
print(f"{'Model':<20} {'Params':<12} {'FP32 (GB)':<12} {'FP16 (GB)':<12} {'INT8 (GB)':<12} {'INT4 (GB)'}")
print("-" * 65)

models = [
    ("distilgpt2",      66_000_000),
    ("GPT-2",          117_000_000),
    ("BERT-Base",      110_000_000),
    ("Llama-3.2-1B",  1_000_000_000),
    ("Llama-3.1-8B",  8_000_000_000),
    ("Llama-3.1-70B", 70_000_000_000),
]

for name, params in models:
    fp32 = params * 4 / (1024**3)
    fp16 = params * 2 / (1024**3)
    int8 = params * 1 / (1024**3)
    int4 = params * 0.5 / (1024**3)
    print(f"{name:<20} {params/1e9:.2f}B       {fp32:<12.2f} {fp16:<12.2f} {int8:<12.2f} {int4:.2f}")

Model Size Comparison for Common LLMs
Model                Params       FP32 (GB)    FP16 (GB)    INT8 (GB)    INT4 (GB)
-----------------------------------------------------------------
distilgpt2           0.07B       0.25         0.12         0.06         0.03
GPT-2                0.12B       0.44         0.22         0.11         0.05
BERT-Base            0.11B       0.41         0.20         0.10         0.05
Llama-3.2-1B         1.00B       3.73         1.86         0.93         0.47
Llama-3.1-8B         8.00B       29.80        14.90        7.45         3.73
Llama-3.1-70B        70.00B       260.77       130.39       65.19        32.60


In [24]:
# ---- Quantization Accuracy vs Compression Trade-off ----

np.random.seed(0)
W = np.random.randn(100).astype(np.float32)   # 100-weight row

results = []
for bits in [32, 16, 8, 4, 2]:
    levels = 2**bits
    q_min  = -(levels // 2)
    q_max  =  (levels // 2) - 1

    S = (W.max() - W.min()) / (q_max - q_min)
    Z = int(round(q_min - W.min() / S))
    Z = int(np.clip(Z, q_min, q_max))

    q_vals = np.clip(np.round(W / S + Z), q_min, q_max).astype(np.int32)
    W_recon = S * (q_vals - Z)

    mse = np.mean((W - W_recon)**2)
    size_bytes = len(W) * bits / 8
    results.append((bits, mse, size_bytes))

print(f"Quantization Trade-off (100 weights)")
print(f"{'Bits':<8} {'MSE (error)':<18} {'Size (bytes)':<15} {'Compression'}")
print("-" * 55)
baseline_size = results[0][2]  # FP32 size
for bits, mse, size in results:
    compression = baseline_size / size
    print(f"  {bits:<6}   {mse:<18.8f}   {size:<15.1f}   {compression:.1f}x")

Quantization Trade-off (100 weights)
Bits     MSE (error)        Size (bytes)    Compression
-------------------------------------------------------
  32       0.23258863           400.0             1.0x
  16       0.00000000           200.0             2.0x
  8        0.00003263           100.0             4.0x
  4        0.00870766           50.0              8.0x
  2        0.19430648           25.0              16.0x


/tmp/ipykernel_2827/792760889.py:16: RuntimeWarning: invalid value encountered in cast
  q_vals = np.clip(np.round(W / S + Z), q_min, q_max).astype(np.int32)


## Part 3d: Mixture of Experts (MoE)

In [25]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ---- Expert Configurations ----
EXPERTS = {
    "math": {
        "system_prompt": """You are a precise Mathematics and Statistics expert.
When given a problem, solve it step-by-step, showing all calculations clearly.
Use LaTeX notation for formulas. Be rigorous and exact.""",
        "temperature": 0.0,
        "emoji": "🔢"
    },
    "code": {
        "system_prompt": """You are a Senior Software Engineer expert.
Write clean, efficient, well-commented code. Always specify the language.
Explain your logic briefly after the code block.""",
        "temperature": 0.1,
        "emoji": "💻"
    },
    "creative": {
        "system_prompt": """You are a Creative Writing expert.
Write with vivid imagery, strong narrative voice, and emotional depth.
Vary sentence structure. Engage the reader's imagination.""",
        "temperature": 0.9,
        "emoji": "✍️"
    },
    "science": {
        "system_prompt": """You are a Science educator expert.
Explain concepts clearly using analogies and real-world examples.
Always ground explanations in empirical evidence.""",
        "temperature": 0.2,
        "emoji": "🔬"
    },
    "general": {
        "system_prompt": "You are a helpful, balanced AI assistant.",
        "temperature": 0.5,
        "emoji": "🤖"
    },
}

print(f"Defined {len(EXPERTS)} experts: {list(EXPERTS.keys())}")

Defined 5 experts: ['math', 'code', 'creative', 'science', 'general']


In [26]:
# ---- Router ----
# Uses temperature=0.0 for deterministic routing
router_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

router_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a query classifier. Classify the user's query into exactly ONE of these categories:
- math       (mathematical calculations, statistics, numerical problems, algebra, equations)
- code       (programming, debugging, software, algorithms, data structures)
- creative   (stories, poems, creative writing, fiction, imagery)
- science    (physics, chemistry, biology, natural phenomena, scientific concepts)
- general    (anything else)

IMPORTANT: Return ONLY the category name. No explanation. No punctuation. Just the one word."""),
    ("human", "{query}")
])

router_chain = router_prompt | router_llm | StrOutputParser()

# Test the router
test_queries = [
    "What is the derivative of x² + 3x?",
    "Write a Python function to reverse a linked list.",
    "Write a short poem about autumn leaves.",
    "How does photosynthesis work?",
    "What time is it in Tokyo?",
]

print("Router Test:")
print("-" * 60)
for q in test_queries:
    category = router_chain.invoke({"query": q}).strip().lower()
    emoji = EXPERTS.get(category, EXPERTS["general"])["emoji"]
    print(f"  {emoji} [{category:<10}] {q}")

Router Test:
------------------------------------------------------------
  🔢 [math      ] What is the derivative of x² + 3x?
  💻 [code      ] Write a Python function to reverse a linked list.
  ✍️ [creative  ] Write a short poem about autumn leaves.
  🔬 [science   ] How does photosynthesis work?
  🤖 [general   ] What time is it in Tokyo?


In [27]:
# ---- MoE Orchestrator ----

def build_expert_chain(expert_key: str):
    """Build a LangChain chain for a specific expert."""
    config = EXPERTS[expert_key]
    llm    = ChatGroq(model="llama-3.1-8b-instant", temperature=config["temperature"])
    prompt = ChatPromptTemplate.from_messages([
        ("system", config["system_prompt"]),
        ("human",  "{query}")
    ])
    return prompt | llm | StrOutputParser()


def moe_process(user_query: str) -> dict:
    """
    Full MoE pipeline:
    1. Router classifies the query
    2. Orchestrator selects the right expert
    3. Expert generates the answer
    """
    # Step 1: Route
    category = router_chain.invoke({"query": user_query}).strip().lower()
    if category not in EXPERTS:
        category = "general"

    # Step 2: Select expert
    expert_chain = build_expert_chain(category)
    emoji = EXPERTS[category]["emoji"]

    # Step 3: Generate
    answer = expert_chain.invoke({"query": user_query})

    return {"category": category, "emoji": emoji, "query": user_query, "answer": answer}


# --- Run the MoE system ---
demo_queries = [
    "What is the integral of sin(x)dx?",
    "Write a recursive function to compute Fibonacci numbers in Python.",
    "How do transformer models use attention mechanisms?",
]

for q in demo_queries:
    result = moe_process(q)
    print(f"{result['emoji']} Expert: {result['category'].upper()}")
    print(f"Query:  {result['query']}")
    print(f"Answer: {result['answer'][:400]}..." if len(result['answer']) > 400 else f"Answer: {result['answer']}")
    print("=" * 70)

🔢 Expert: MATH
Query:  What is the integral of sin(x)dx?
Answer: To find the integral of $\sin(x)$, we can use the following formula:

$$\int \sin(x) \, dx = -\cos(x) + C$$

where $C$ is the constant of integration.

To derive this formula, we can use the following steps:

1. Recall the definition of the derivative of the cosine function:

$$\frac{d}{dx} \cos(x) = -\sin(x)$$

2. Integrate both sides of the equation with respect to $x$:

$$\int \frac{d}{dx} \cos...
💻 Expert: CODE
Query:  Write a recursive function to compute Fibonacci numbers in Python.
Answer: **Fibonacci Recursive Function in Python**
```python
def fibonacci(n: int) -> int:
    """
    Compute the nth Fibonacci number using recursion.

    Args:
    n (int): The position of the Fibonacci number to compute.

    Returns:
    int: The nth Fibonacci number.

    Raises:
    ValueError: If n is a negative integer.
    """

    # Base case: Fibonacci numbers are 0 and 1
    if n <= 0:
     ...
💻 Expert: CODE
Query:  How do

In [28]:
# ---- Demonstrating Expert Specialization: Same query, different experts ----
# This shows WHY routing matters — the same question gets very different answers
# depending on which "expert" handles it.

query = "Explain recursion."

print(f"Query: '{query}'")
print("=" * 70)
for expert_key in ["code", "creative", "science"]:
    chain  = build_expert_chain(expert_key)
    answer = chain.invoke({"query": query})
    emoji  = EXPERTS[expert_key]["emoji"]
    print(f"\n{emoji} {expert_key.upper()} Expert:")
    print(answer[:300] + "..." if len(answer) > 300 else answer)
    print("-" * 70)

Query: 'Explain recursion.'

💻 CODE Expert:
**Recursion in Programming**

Recursion is a fundamental concept in programming where a function calls itself repeatedly until it reaches a base case that stops the recursion. This technique allows us to solve problems that have a recursive structure, where a problem...
----------------------------------------------------------------------

✍️ CREATIVE Expert:
In the depths of a mystical forest, where trees stretched towards the sky like nature's own cathedral, a secret lay hidden. A secret that unraveled the very fabric of computation, a secret that twisted and turned like a serpent through the code. This was recursion, a phenomenon that defied the linea...
----------------------------------------------------------------------

🔬 SCIENCE Expert:
Recursion is a fundamental concept in computer science and mathematics that can be a bit tricky to grasp at first, but don't worry, I'll break it down using analogies and real-world examples.

**What